In [1]:
import os
import json
import math
import warnings
from dataclasses import dataclass
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


# 0) USER CONFIG


In [2]:
DATA_PATH = "/content/Loan_default.csv"
OUTPUT_DIR = "/content/woe_prepared_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET = "Default"
ID_COLS = ["LoanID"]

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_DEV = 0.25  # to get 60/20/20

MAX_BINS = 20
IV_THRESHOLD = 0.05
MIN_FEATURES = 3

# ChiMerge binning is not used, instead Monotonic binning. For numerical variables, neighboring
# bins are therefore merged until the observed bad rates follow a monotonic pattern where possible
ENFORCE_MONOTONIC_BINNING = True
MIN_MONOTONIC_BINS = 3


# 1) HELPER FUNCTIONS


In [3]:
# Creates quantile-based bin edges for numerical variables
def make_numeric_bins(series: pd.Series, max_bins: int = 10):
    s = series.dropna()
    if s.nunique() < 2:
        return None

    q = np.linspace(0, 1, max_bins + 1)
    edges = np.unique(np.quantile(s, q))
    if len(edges) < 3:
        return None

    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges

# Applies predefined numerical bin edges; handles missing values as a separate bin
def apply_numeric_bins(series: pd.Series, edges):
    if edges is None:
        return pd.Series(["ALL"] * len(series), index=series.index, dtype="object")

    binned = pd.cut(series, bins=edges, include_lowest=True)
    binned = binned.astype("object")
    binned = binned.where(~series.isna(), "MISSING")
    return binned.astype(str)

# Merges neighboring numerical bins until the bad-rate pattern is monotonic where possible
def merge_bins_until_monotone(
    series: pd.Series,
    target: pd.Series,
    initial_edges: np.ndarray,
    min_bins: int = 3
):

    if initial_edges is None:
        return None

    edges = list(initial_edges)

    while True:
        binned = apply_numeric_bins(series, np.array(edges))
        table = compute_woe_table(binned, target)

        # remove special bins
        tab = table[~table["bin"].isin(["MISSING", "ALL"])].copy()

        if len(tab) < min_bins:
            break

        bad_rates = tab["bad_rate"].values

        is_increasing = np.all(np.diff(bad_rates) >= -1e-12)
        is_decreasing = np.all(np.diff(bad_rates) <= 1e-12)

        if is_increasing or is_decreasing:
            break

        # Find adjacent pair with smallest bad-rate difference
        diffs = np.abs(np.diff(bad_rates))
        merge_idx = int(np.argmin(diffs))

        # Remove internal edge between the two bins
        # merge_idx refers to adjacent bins, so remove edge at merge_idx + 1
        if len(edges) <= min_bins + 1:
            break

        del edges[merge_idx + 1]

    return np.array(edges)

# Converts categorical variables to string format and replaces missing values (MISSING as own category) --> in scorecards missings are a category, no imputation
def prepare_categorical(series: pd.Series):
    return series.astype("object").fillna("MISSING").astype(str)

# Computes WoE, IV components, and bad-rate statistics for each bin of a single variable
def compute_woe_table(bin_series: pd.Series, target: pd.Series, smoothing: float = 0.5):
    tmp = pd.DataFrame({"bin": bin_series.astype(str), "target": np.asarray(target).astype(int)})

    grouped = tmp.groupby("bin", dropna=False)["target"].agg(["count", "sum"]).rename(columns={"sum": "bads"})
    grouped["goods"] = grouped["count"] - grouped["bads"]

    total_goods = grouped["goods"].sum()
    total_bads = grouped["bads"].sum()
    n_bins = len(grouped)

    grouped["dist_goods"] = (grouped["goods"] + smoothing) / (total_goods + smoothing * n_bins)
    grouped["dist_bads"] = (grouped["bads"] + smoothing) / (total_bads + smoothing * n_bins)
    grouped["woe"] = np.log(grouped["dist_goods"] / grouped["dist_bads"])
    grouped["iv_component"] = (grouped["dist_goods"] - grouped["dist_bads"]) * grouped["woe"]
    grouped = grouped.reset_index()
    grouped["bad_rate"] = grouped["bads"] / grouped["count"]
    return grouped

# Transforms binned variable values into their corresponding WoE values
def transform_to_woe(bin_series: pd.Series, woe_table: pd.DataFrame):
    woe_map = dict(zip(woe_table["bin"].astype(str), woe_table["woe"]))
    return bin_series.astype(str).map(woe_map).fillna(0.0)

# Checks whether bad rates across numerical bins consistently increase or decrease
# This is used as a diagnostic to eval whether the variable shows a stable risk pattern, special bins like 'MISSING' and 'ALL' (--> if no useful binning is possible) are ignored.
def monotonicity_flag_numeric(woe_table):
    mask = woe_table["bin"].isin(["MISSING", "ALL"]) == False
    rates = woe_table.loc[mask, "bad_rate"].astype(float).values

    if len(rates) < 3:
        return "not_applicable"

    if np.all(np.diff(rates) >= -1e-12):
        return "monotone_increasing"

    if np.all(np.diff(rates) <= 1e-12):
        return "monotone_decreasing"

    return "non_monotone"


@dataclass
class WoETransformer:
    bin_definitions: Dict[str, Dict[str, Any]]
    woe_tables: Dict[str, pd.DataFrame]
    iv_df: pd.DataFrame
    selected_features: List[str]
    feature_types: Dict[str, str]
    feature_order: List[str]
    config: Dict[str, Any]

def fit_woe_transformer(
    X_train_raw,
    y_train,
    max_bins=10,
    iv_threshold=0.02,
    min_features=3
):
    # Feature-Types
    numeric_features = X_train_raw.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    feature_order = list(X_train_raw.columns)

    bin_definitions = {}
    woe_tables = {}
    iv_rows = []

    # numeric features
    for col in numeric_features:
        edges = make_numeric_bins(X_train_raw[col], max_bins)

        if ENFORCE_MONOTONIC_BINNING and edges is not None:
            edges = merge_bins_until_monotone(
                X_train_raw[col],
                y_train,
                edges,
                MIN_MONOTONIC_BINS
            )

        bins = apply_numeric_bins(X_train_raw[col], edges)
        table = compute_woe_table(bins, y_train)

        bin_definitions[col] = {"type": "numeric", "edges": edges}
        woe_tables[col] = table

        iv_rows.append({
            "feature": col,
            "iv": float(table["iv_component"].sum())
        })

    # categorical features
    for col in categorical_features:
        bins = prepare_categorical(X_train_raw[col])
        table = compute_woe_table(bins, y_train)

        bin_definitions[col] = {"type": "categorical"}
        woe_tables[col] = table

        iv_rows.append({
            "feature": col,
            "iv": float(table["iv_component"].sum())
        })

    # IV-tabels
    iv_df = pd.DataFrame(iv_rows).sort_values("iv", ascending=False)

    # Feature selection
    selected_features = iv_df[iv_df["iv"] >= iv_threshold]["feature"].tolist()

    if len(selected_features) < min_features:
        selected_features = feature_order

    return {
        "bin_definitions": bin_definitions,
        "woe_tables": woe_tables,
        "iv_df": iv_df,
        "selected_features": selected_features,
        "feature_order": feature_order
    }
# Fit WoE on training data and transform it
def fit_and_transform_woe(X_train, y_train, max_bins=10, iv_threshold=0.02, min_features=3):

    # learn WoE transformation from training data
    transformer = fit_woe_transformer(
        X_train,
        y_train,
        max_bins=max_bins,
        iv_threshold=iv_threshold,
        min_features=min_features
    )

    # apply transformation to training data
    X_train_woe = transform_woe_data(transformer, X_train)

    return transformer, X_train_woe


# Apply WoE transformation to new data
def transform_woe_data(transformer, X):

    X_woe = pd.DataFrame(index=X.index)

    for col in transformer["feature_order"]:

        meta = transformer["bin_definitions"][col]

        if meta["type"] == "numeric":
            bins = apply_numeric_bins(X[col], meta["edges"])
        else:
            bins = prepare_categorical(X[col])

        X_woe[col] = transform_to_woe(bins, transformer["woe_tables"][col])

    return X_woe

# Convert bin edges to JSON-compatible format
# Replace inf / -inf with strings, since JSON does not support them
# inf and -inf are used to ensure that all values fall into a bin
def edges_to_jsonable(edges):
    if edges is None:
        return None

    result = []
    for x in edges:
        if x == float("inf"):
            result.append("inf")
        elif x == float("-inf"):
            result.append("-inf")
        else:
            result.append(float(x))

    return result



# 2) LOAD DATA

In [4]:
df = pd.read_csv(DATA_PATH)

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found")

# Drop rows with missing target values
df = df.dropna(subset=[TARGET]).copy()

y = df[TARGET].astype(int)
X = df.drop(columns=ID_COLS + [TARGET], errors="ignore")

# 3) SPLIT RAW DATA

In [5]:
# train and test split
X_dev_full, X_test, y_dev_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

# train and validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_dev_full,
    y_dev_full,
    test_size=VAL_SIZE_WITHIN_DEV,
    random_state=SEED,
    stratify=y_dev_full,
)

# Reset indices to avoid alignment issues after loading
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [6]:
# Renaming the splits for AutoML
train_raw = X_train.copy()
train_raw[TARGET] = y_train.values

val_raw = X_val.copy()
val_raw[TARGET] = y_val.values

test_raw = X_test.copy()
test_raw[TARGET] = y_test.values

train_final_raw = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_final_raw = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
train_final_raw[TARGET] = y_train_final_raw.values

test_final_raw = X_test.copy()
test_final_raw[TARGET] = y_test.values

# 4) Clipping on numeric features

In [7]:
numeric_features = [
    c for c in X_train.columns
    if pd.api.types.is_numeric_dtype(X_train[c])
]

clip_bounds = {}

clipping_summary = []

# Determine limits on Train

for col in numeric_features:

    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    n_lower = (X_train[col] < lower).sum()
    n_upper = (X_train[col] > upper).sum()

    clip_bounds[col] = {
        "lower": lower,
        "upper": upper
    }

    clipping_summary.append({
        "feature": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "n_clipped_lower": n_lower,
        "n_clipped_upper": n_upper
    })

# Apply to all splits

for col in numeric_features:

    lower = clip_bounds[col]["lower"]
    upper = clip_bounds[col]["upper"]

    X_train[col] = X_train[col].clip(lower, upper)
    X_val[col] = X_val[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

# Save clipping information

clipping_summary = pd.DataFrame(clipping_summary)

clipping_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "clipping_summary.csv"
    ),
    index=False
)

print(clipping_summary)

          feature  lower_bound  upper_bound  n_clipped_lower  n_clipped_upper
0             Age        18.00        69.00                0                0
1          Income     16340.12    148658.00             1533             1530
2      LoanAmount      7407.00    247538.94             1532             1533
3     CreditScore       305.00       844.00             1419             1389
4  MonthsEmployed         1.00       118.00             1271             1250
5  NumCreditLines         1.00         4.00                0                0
6    InterestRate         2.23        24.78             1497             1502
7        LoanTerm        12.00        60.00                0                0
8        DTIRatio         0.11         0.89              963              942


# 4) FIT WoE AND TRANSFORM DATA

In [ ]:

# Fit WoE transformation on training data only
woe_transformer_dev, X_train_woe = fit_and_transform_woe(X_train, y_train)

# use the same WoE transformation to validation and test data, no fitting
X_val_woe = transform_woe_data(woe_transformer_dev, X_val)
# Test set transformed with WoE transformer fitted on training data only, used for model selection / validation-stage evaluation
X_test_woe = transform_woe_data(woe_transformer_dev, X_test)

# Add target variable back to the transformed datasets
train_woe = X_train_woe.copy()
train_woe[TARGET] = y_train.values

val_woe = X_val_woe.copy()
val_woe[TARGET] = y_val.values

test_woe = X_test_woe.copy()
test_woe[TARGET] = y_test.values



# Combine training and validation data, as this allows the final model to learn from more data
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Fit WoE transformation again on the full training data (train + validation)
woe_transformer_final, X_train_final_woe = fit_and_transform_woe(X_train_final, y_train_final)

# Apply transformation to test data
# Test set transformed with WoE transformer fitted on train + validation data, used for final model evaluation
X_test_final_woe = transform_woe_data(woe_transformer_final, X_test)

# Add target variable back
train_final_woe = X_train_final_woe.copy()
train_final_woe[TARGET] = y_train_final.values

test_final_woe = X_test_final_woe.copy()
test_final_woe[TARGET] = y_test.values




# 5. REPORTING

In [ ]:

# Basic dataset information before WoE transformation
dataset_summary = pd.DataFrame({
    "field": ["n_rows", "n_columns", "target", "default_rate"],
    "value": [len(df), df.shape[1], TARGET, float(y.mean())],
})

# Split summary for the model-selection setup
# Shows the number of observations and default rate in train, validation, and test data
split_summary = pd.DataFrame({
    "sample": ["train", "val", "test"],
    "n_obs": [len(X_train), len(X_val), len(X_test)],
    "default_rate": [float(y_train.mean()), float(y_val.mean()), float(y_test.mean())],
})

# IV summary based on the WoE transformer fitted on training data only
# --> The "selected column" means which features passed the IV threshold (0.02)
iv_summary = woe_transformer_dev["iv_df"].copy()
iv_summary["selected"] = iv_summary["feature"].isin(woe_transformer_dev["selected_features"])

# Full WoE table for all features and bins
woe_table_all = pd.concat([
    table.assign(feature=feature)
    for feature, table in woe_transformer_dev["woe_tables"].items()
], ignore_index=True)

# Metadata for reproducibility of the model-selection WoE transformation
metadata = {
    "target": TARGET,
    "n_rows": int(len(df)),
    "n_features": int(X.shape[1]),
    "selected_features": woe_transformer_dev["selected_features"],
    "woe_fit_sample": "train",
    "validation_and_test_usage": "transformed only, not fitted",
}

# IV summary based on the final WoE transformer
# --> transformer was fitted on train + validation data
iv_summary_final = woe_transformer_final["iv_df"].copy()
iv_summary_final["selected"] = iv_summary_final["feature"].isin(
    woe_transformer_final["selected_features"]
)

# Full WoE table for the final WoE transformation
woe_table_all_final = pd.concat([
    table.assign(feature=feature)
    for feature, table in woe_transformer_final["woe_tables"].items()
], ignore_index=True)

# Metadata for the final WoE transformation
metadata_final = {
    "target": TARGET,
    "n_rows_fit_sample": int(len(X_train_final)),
    "n_features": int(X_train_final.shape[1]),
    "selected_features": woe_transformer_final["selected_features"],
    "woe_fit_sample": "train + validation",
    "test_usage": "test was transformed only, not fitted",
}

# 6. SAVE OUTPUT

In [ ]:

# Save the CSVs for AutoML
train_raw.to_csv(os.path.join(OUTPUT_DIR, "train_raw.csv"), index=False)
val_raw.to_csv(os.path.join(OUTPUT_DIR, "val_raw.csv"), index=False)
test_raw.to_csv(os.path.join(OUTPUT_DIR, "test_raw.csv"), index=False)


# Save WoE-transformed datasets for model selection
# The WoE transformer was fitted on train only; validation and test were only transformed
train_woe.to_csv(os.path.join(OUTPUT_DIR, "train_woe.csv"), index=False)
val_woe.to_csv(os.path.join(OUTPUT_DIR, "val_woe.csv"), index=False)
test_woe.to_csv(os.path.join(OUTPUT_DIR, "test_woe.csv"), index=False)

# Save WoE-transformed datasets for final model training and evaluation
# Here, WoE was fitted again on train + validation; test was only transformed
train_final_woe.to_csv(os.path.join(OUTPUT_DIR, "train_final_woe.csv"), index=False)
test_final_woe.to_csv(os.path.join(OUTPUT_DIR, "test_final_woe.csv"), index=False)

# Save documentation tables for the model-selection WoE transformation
dataset_summary.to_csv(os.path.join(OUTPUT_DIR, "dataset_summary.csv"), index=False)
split_summary.to_csv(os.path.join(OUTPUT_DIR, "split_summary.csv"), index=False)
iv_summary.to_csv(os.path.join(OUTPUT_DIR, "iv_summary.csv"), index=False)
woe_table_all.to_csv(os.path.join(OUTPUT_DIR, "woe_table_all.csv"), index=False)

# Save metadata for reproducibility
with open(os.path.join(OUTPUT_DIR, "woe_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Save documentation tables for the final WoE transformation
iv_summary_final.to_csv(os.path.join(OUTPUT_DIR, "iv_summary_final.csv"), index=False)
woe_table_all_final.to_csv(os.path.join(OUTPUT_DIR, "woe_table_all_final.csv"), index=False)

# Save metadata as JSON for reproducibility and documentation
with open(os.path.join(OUTPUT_DIR, "woe_metadata_final.json"), "w", encoding="utf-8") as f:
    json.dump(metadata_final, f, indent=2)